# MATH-N V1 Demo

**Verifier-Driven Numerical Mathematics**

This notebook clones the MATH-N repository, installs dependencies, generates the synthetic V1 dataset, runs a basic sanity check, runs the full V1 benchmark (SciPy and SymPy candidate generators against independent numerical verifiers), and displays aggregate metrics and an example trajectory.

No GPU is required.

## 1. Clone the repository

In [ ]:
# Replace with your fork/remote if needed.
!git clone https://github.com/your-org/math-n.git
%cd math-n

## 2. Install dependencies

In [ ]:
!pip install -q -e .

## 3. Generate the V1 development dataset

In [ ]:
!python -m mathn.cli generate-dataset --dataset datasets/v1 --problems-per-family 10

## 4. Basic numerical sanity test

Demonstrate the three core behaviors of the verifier-driven loop on a single ODE problem:

1. A correct candidate is **VERIFIED**.
2. An intentionally incorrect candidate is **REJECTED**, with structured numerical feedback captured.
3. A classical generator that starts loose and tightens its tolerance **retries** until it converges.

In [ ]:
import sys
sys.path.insert(0, 'src')

import mathn.problems, mathn.generators
from mathn.problems.ode import ODEProblemGenerator
from mathn.verifiers.ode_verifier import ODEVerifier
from mathn.generators.scipy_generator import ScipyGenerator
from mathn.core.models import Candidate
from mathn.core.runner import RetryController, RetryConfig

problem = ODEProblemGenerator().generate(seed=20260101, index=0)
verifier = ODEVerifier()
print('Problem:', problem.public['equation'], '| query t =', problem.public['t_query'])

# 1. Correct candidate -> VERIFIED
correct = Candidate(problem_id=problem.problem_id, generator='oracle', attempt=1,
                     result={'y_t_query': problem.protected['reference_y']})
print('\nCorrect candidate  ->', verifier.verify(problem, correct).status)

# 2. Intentionally wrong candidate -> REJECTED with feedback
wrong = Candidate(problem_id=problem.problem_id, generator='buggy', attempt=1,
                   result={'y_t_query': problem.protected['reference_y'] + 1.0})
bad_result = verifier.verify(problem, wrong)
print('Incorrect candidate ->', bad_result.status, '| feedback:', bad_result.details)

# 3. Retry behavior with the real SciPy generator
controller = RetryController(ScipyGenerator(), verifier, RetryConfig(max_attempts=5))
traj = controller.run(problem)
print('\nRetry trajectory: final_status =', traj.final_status, '| attempts_used =', traj.attempts_used)
for a in traj.attempts:
    print('  attempt', a.attempt, '->', a.verification.status, '| abs_error =', a.verification.details.get('absolute_error'))

## 5. Run the full V1 benchmark (SciPy + SymPy, all four problem families)

In [ ]:
!python experiments/run_v1.py

## 6. Display aggregate metrics

In [ ]:
import json
report = json.load(open('results/v1_report.json'))

for gen_name, gen_report in report['generators'].items():
    m = gen_report['metrics']
    print(f"--- {gen_name} ---")
    print(f"  final success rate:        {m['final_success_rate']:.1%}")
    print(f"  first-attempt success rate: {m['first_attempt_success_rate']:.1%}")
    print(f"  average attempts:          {m['average_attempts']:.2f}")
    print(f"  mean absolute error:       {m['mean_absolute_error']:.3g}")
    print(f"  malformed candidates:      {m['malformed_candidate_count']}")
    print()

## 7. Inspect one trajectory in detail

In [ ]:
import glob

# Find a trajectory that needed more than one attempt, to show retry behavior.
candidates = sorted(glob.glob('trajectories/scipy/mathn_ode_*.json'))
example_path = None
for path in candidates:
    d = json.load(open(path))
    if d['attempts_used'] > 1:
        example_path = path
        break

example = json.load(open(example_path))
print('Trajectory:', example_path)
print(json.dumps(example, indent=2))

## 8. Failure analysis

In [ ]:
for gen_name, gen_report in report['generators'].items():
    fa = gen_report['failure_analysis']
    print(f"--- {gen_name} ---")
    print('  category counts:', fa['category_counts'])
    print('  unsolved problems:', fa['num_unsolved'])
    print('  converged after retry:', fa['num_converged_after_retry'])
    print()

## 9. Results are already saved

`results/v1_report.json` contains the full metrics + failure analysis for both generators, and `trajectories/<generator>/*.json` contains every individual attempt for every problem -- nothing here needs a separate save step.